In [0]:
from delta.tables import *

stage_table_name = 'incremental_load.default.orders_stage'
target_table_name = 'incremental_load.default.orders_target'

In [0]:
#Read data from stage Table
stage_df = spark.read.table(stage_table_name)

In [0]:
# Create target table schema if not exist
if not spark.catalog.tableExists('target_table_name'):
    stage_df.write.format('delta').saveAsTable('target_table_name')
else:
    # Read target table
    target_table = DeltaTable.forName(spark,target_table_name)

    # Merge Condition
    merge_condition = "stage.tracking_num = target.tracking_num"

    # Perform Merge Operation
    target_table.alias('target')\
        .merge(stage_df.alias('stage'),merge_condition)\
        .whenMatchedDelete()\
        .execute()

    stage_df.write.format('delta').mode('append').saveAsTable(target_table_name)